In [ ]:
import pandas as pd
import requests
import chardet
import string
from rapidfuzz import fuzz
import time

# Detect file encoding and load CSV
with open("schools_output.csv", "rb") as f:
    result = chardet.detect(f.read())

df = pd.read_csv("schools_output.csv", encoding=result['encoding'])

# Normalize columns: lowercase, strip spaces, remove spaces inside names
df.columns = df.columns.str.strip().str.lower().str.replace(" ", "")

# Check required columns
required_cols = {"name", "latitude", "longitude"}
missing = required_cols - set(df.columns)
if missing:
    raise ValueError(f"Missing columns: {missing}")

API_KEY = "YOUR_GOOGLE_MAPS_API_KEY"  # Replace with your actual Google Maps API key

def reverse_geocode(lat, lon):
    url = f"https://maps.googleapis.com/maps/api/geocode/json?latlng={lat},{lon}&key={API_KEY}"
    try:
        response = requests.get(url, timeout=5)
        response.raise_for_status()
        data = response.json()
        if data['status'] == 'OK' and len(data['results']) > 0:
            return data['results'][0]['formatted_address']
        else:
            return ""
    except Exception as e:
        print(f"Error with lat={lat}, lon={lon}: {e}")
        return ""

def clean_text(text):
    if not text or pd.isna(text):
        return ""
    text = str(text).lower().strip()
    return text.translate(str.maketrans('', '', string.punctuation))



def fuzzy_name_match(school_name, address, threshold=60):
    if not school_name or not address:
        return False
    score = fuzz.partial_ratio(school_name, address)
    return score >= threshold

def keyword_match(school_name, address, min_words=2):
    school_words = set(school_name.split())
    address_words = set(address.split())
    common = school_words.intersection(address_words)
    return len(common) >= min_words

def combined_match(school_name, address):
    # Try fuzzy OR keyword match, tweak as needed
    return fuzzy_name_match(school_name, address, threshold=60) or keyword_match(school_name, address, min_words=2)


# Reverse geocode addresses
df["reverse_geocode"] = df.apply(lambda row: reverse_geocode(row["latitude"], row["longitude"]), axis=1)

# Clean texts for matching
df["clean_name"] = df["name"].apply(clean_text)
df["clean_reverse_geocode"] = df["reverse_geocode"].apply(clean_text)

# Fuzzy match school name vs reverse geocoded address
df["name_match"] = df.apply(
    lambda row: combined_match(row["clean_name"], row["clean_reverse_geocode"]),
    axis=1
)



# Debug print for mismatches
print("\nSchools with no name match:\n")
for idx, row in df.iterrows():
    if not row["name_match"]:
        print("School name:", row["name"])
        print("Reverse Geocode:", row["reverse_geocode"])
        print("Clean name:", row["clean_name"])
        print("Clean reverse:", row["clean_reverse_geocode"])
        print("---")

# Save output CSV
df.to_csv("validated_schools.csv", index=False)
print("✅ Validation complete. Output saved to validated_schools.csv")
